## Approach 1: Brute Force

For each city in `dim_city`, find the closest forecast point from `silver_hourly_forecast`.

**How it works:**

1. **Cross join** every city against every distinct forecast coordinate (no filtering by proximity up front).
2. Compute the **haversine distance** (in km) between each city and each forecast point using the `haversine_km` helper defined below.
3. **Rank** the forecast points by distance for each city (window partitioned by city, ordered by `distance_km`) and keep only the **closest one** (`qualify rank = 1`).
4. Attach the chosen forecast point's hourly weather (temperature, precipitation, wind, weather code, etc.) to the city.

This is simple and easy to reason about, but it does O(cities × forecast points) work — fine for small-to-moderate table sizes, but potentially expensive as either side grows.


In [0]:
from typing import Annotated

from pyspark.sql import Column, functions as F
from pyspark.sql.types import DoubleType

DoubleCol = Annotated[Column, DoubleType()]


def haversine_km(
    lat1: DoubleCol,
    lon1: DoubleCol,
    lat2: DoubleCol,
    lon2: DoubleCol,
) -> DoubleCol:
    earth_radius_km = 6371.0

    lat1_rad = F.radians(lat1)
    lon1_rad = F.radians(lon1)
    lat2_rad = F.radians(lat2)
    lon2_rad = F.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = (
        F.pow(F.sin(dlat / 2), 2)
        + F.cos(lat1_rad) * F.cos(lat2_rad) * F.pow(F.sin(dlon / 2), 2)
    )

    return 2 * earth_radius_km * F.asin(F.least(F.lit(1.0), F.sqrt(a)))

In [0]:
city = spark.table("workspace.open_meteo.dim_city").alias("C")
fc = spark.table("workspace.open_meteo.silver_hourly_forecast").alias("F")

result = city.crossJoin(fc).select(
    F.col("C.name"),
    F.col("C.country"),
    F.col("C.latitude").alias("city_lat"),
    F.col("C.longitude").alias("city_lon"),
    F.col("F.latitude").alias("fc_lat"),
    F.col("F.longitude").alias("fc_lon"),
    F.col("F.forecast_time"),
    F.col("F.temperature_2m"),
    F.col("F.apparent_temperature"),
    F.col("F.precipitation"),
    F.col("F.wind_speed_10m"),
    F.col("F.weather_code"),
    F.col("F.fetched_at"),
    haversine_km(
        F.col("C.latitude"),
        F.col("C.longitude"),
        F.col("F.latitude"),
        F.col("F.longitude"),
    ).alias("distance_km"),
)

In [0]:
result.createOrReplaceTempView("city_distances")

In [0]:
%sql
select *,
rank() over (partition by name, country, city_lat, city_lon order by distance_km) as rank
from city_distances
qualify rank = 1
order by name, country, city_lat, city_lon, distance_km

## Approach 2: Distinct Coordinates Then Join

Improves on the brute-force approach by shrinking the cross join to only the **distinct** (`latitude`, `longitude`) pairs in `silver_hourly_forecast` before computing distances.

**How it works:**

1. **Deduplicate** `silver_hourly_forecast` to its distinct coordinates — every forecast row for a given point shares the same lat/lon, so one row per point is enough for distance matching.
2. **Cross join** each city against only these distinct points and compute the **haversine distance** with `haversine_km`.
3. **Rank** by distance per city and keep the **closest point** (`qualify rank = 1`), yielding a city → forecast-point mapping table.
4. **Join** that small mapping back to the full `silver_hourly_forecast` on the chosen coordinates to attach the hourly weather.

The cross join is now O(cities × distinct points) instead of O(cities × forecast rows), which can be a large win when each forecast point has many hourly rows.

In [0]:
locs = fc.select(
    F.col("latitude").alias("fc_lat"),
    F.col("longitude").alias("fc_lon"),
).distinct()

nearest = (
    city.crossJoin(F.broadcast(locs))
    .withColumn(
        "distance_km",
        haversine_km(
            F.col("C.latitude"), F.col("C.longitude"),
            F.col("fc_lat"), F.col("fc_lon"),
        ),
    )
    .groupBy("C.name", "C.country", "C.latitude", "C.longitude")
    .agg(
        F.min_by(F.struct("fc_lat", "fc_lon"), "distance_km").alias("loc"),
        F.min("distance_km").alias("distance_km"),
    )
    .select(
        "name", "country",
        F.col("latitude").alias("city_lat"),
        F.col("longitude").alias("city_lon"),
        "loc.fc_lat", "loc.fc_lon", "distance_km",
    )
)

In [0]:
result = nearest.join(
    fc,
    (nearest.fc_lat == fc.latitude) & (nearest.fc_lon == fc.longitude),
)
display(result)

## Approach 3: H3 Grid Join (Scalable)

The most sophisticated and scalable approach — avoids the cross join entirely by matching cities to forecast points through Uber's **H3 hexagonal spatial index**, using a broadcast hash join on H3 cell IDs.

**How it works:**

1. **Deduplicate** `silver_hourly_forecast` to its distinct coordinates, then **explode each point into its H3 k-ring neighbors** (`h3_kring(..., K)`) at resolution 5 (~12.8 km cells). Since cities can sit in a different H3 cell than their nearest forecast point, expanding each point into its surrounding ring guarantees the match is still found.
2. **Index the cities** — assign each row of `dim_city` its H3 cell (`h3_longlatash3`) at the same resolution. `dim_city` stays at 1x rows; only the small distinct-locations side is expanded 7x (self + 6 neighbors).
3. **Broadcast hash join** the exploded locations against the cities on the cell ID — a zero-shuffle broadcast join, since the exploded coordinate table is small enough to broadcast.
4. **Exact distance tie-breaker** — compute `st_distancesphere` for each candidate pair, then use `min_by` to pick the truly closest forecast point per city (the hex grid narrows candidates; the haversine/sphere distance decides the winner).

This turns an O(cities × points) cross join into an O(cities × ~7) indexed lookup, making it the best choice as either table grows large.

```text
dim_city (1x rows, H3 cell per city)  ──broadcast──▶  exploded distinct locations (7x, H3 neighbors)
                          └── min_by(distance_km) ──▶ exact nearest forecast point per city
```

In [0]:
from pyspark.sql import functions as F

city = spark.table("workspace.open_meteo.dim_city").alias("C")
fc = spark.table("workspace.open_meteo.silver_hourly_forecast").alias("F")

RES = 5  # ~12.8 km guaranteed 1-ring radius (covers 3–6 km API grid snap)
K = 1

# 1. Explode the smaller unique grid-locations table (7x) and broadcast it
locs_c = (
    fc.select(
        F.col("latitude").alias("fc_lat"),
        F.col("longitude").alias("fc_lon"),
    )
    .distinct()
    .withColumn(
        "nbr",
        F.explode(F.expr(f"h3_kring(h3_longlatash3(fc_lon, fc_lat, {RES}), {K})")),
    )
)

# 2. Keep dim_city at 1x rows
city_c = (
    spark.table("workspace.open_meteo.dim_city")
    .withColumn("cell", F.expr(f"h3_longlatash3(longitude, latitude, {RES})"))
)

# 3. Broadcast hash join (zero shuffle on dim_city) + exact distance tie-breaker
nearest = (
    city_c.join(F.broadcast(locs_c), city_c.cell == locs_c.nbr, how="inner")
    .withColumn(
        "distance_km",
        F.expr(
            "st_distancesphere(st_point(longitude, latitude), st_point(fc_lon, fc_lat)) / 1000.0"
        ),
    )
    .groupBy("name", "country", "latitude", "longitude")  # use ["city_id"] if available
    .agg(
        F.min_by(
            F.struct("distance_km", "fc_lat", "fc_lon"),
            "distance_km",
        ).alias("best")
    )
    .select(
        "name",
        "country",
        F.col("latitude").alias("city_lat"),
        F.col("longitude").alias("city_lon"),
        "best.fc_lat",
        "best.fc_lon",
        "best.distance_km",
    )
)

In [0]:
result = nearest.join(
    fc,
    (nearest.fc_lat == fc.latitude) & (nearest.fc_lon == fc.longitude),
)
display(result)

In [0]:
html = open("h3_spatial_join_on_3d_earth.html").read()
displayHTML(html)